# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json


In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import numpy as np

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Fetch the dataset's metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets available in the dataset, with their @id and name
print("Record sets (with their '@id' and name):")
for record_set in metadata.record_sets:
    print(f"  @id: {record_set.id} | Name: {record_set.name}")

# Let's print the fields and columns of each record set
for record_set in metadata.record_sets:
    print(f"\nRecordSet '@id': {record_set.id}  Name: {record_set.name}")
    for field in record_set.fields:
        print(f"  Field @id: {field.id} | Name: {field.name} | Data type: {field.data_type}")
        if hasattr(field, 'column') and field.column:
            if isinstance(field.column, list):
                for column in field.column:
                    print(f"    Column @id: {column.id} | Name: {getattr(column, 'name', '(no name)')}")
            else:
                column = field.column
                print(f"    Column @id: {column.id} | Name: {getattr(column, 'name', '(no name)')}")


## 3. Data Extraction
Load data from one or more record sets into pandas DataFrames. Use the record set and field `@id`s from the overview above.

In [ ]:
# Define the record sets to extract
# Replace these with the main table(s) you want to analyze, according to the Data Overview above.
# For demonstration, we list all record sets by their @id.
record_sets = [record_set.id for record_set in metadata.record_sets]
dataframes = {}

for record_set_id in record_sets:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded DataFrame for RecordSet '@id': {record_set_id}, shape: {df.shape}")

# For the main exploration, choose one record set (the primary data table)
if len(dataframes) > 0:
    main_record_set_id = list(dataframes.keys())[0]
    main_df = dataframes[main_record_set_id]
    print(f"\nColumns available in main DataFrame (RecordSet @id: {main_record_set_id}):\n{main_df.columns.tolist()}")
    display(main_df.head())
else:
    print("No records could be loaded from any record set.")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This often involves removing outliers, transforming data distributions, or grouping data by key attributes.

In [ ]:
if len(dataframes) > 0:
    # Choose numeric fields for demonstration (replace with actual field names from the overview, e.g. 'cr:coverage' or 'cr:molecular_weight' or similar)
    df = main_df
    numeric_fields = df.select_dtypes(include=np.number).columns.tolist()
    if len(numeric_fields) == 0:
        print("No numeric fields found for EDA.")
    else:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean()  # Split at mean for demo
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with '{numeric_field}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize the numeric field for filtered records
        filtered_df[numeric_field + "_normalized"] = (
            filtered_df[numeric_field] - filtered_df[numeric_field].mean()
        ) / (filtered_df[numeric_field].std() + 1e-9)  # avoid div0
        print(f"\nNormalized '{numeric_field}' for filtered records:")
        display(filtered_df[[numeric_field, numeric_field + "_normalized"]].head())

        # Group by a categorical field (if any exist)
        groupable_fields = df.select_dtypes(include='object').columns.tolist()
        if len(groupable_fields) > 0:
            group_field = groupable_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"\nGrouped mean '{numeric_field}' by '{group_field}':")
            display(grouped_df.head())
        else:
            print("No categorical fields found to group on.")
else:
    print("No DataFrames loaded for EDA.")


## 5. Visualization
Visualize the distribution of a numeric variable, or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt

if len(dataframes) > 0 and len(main_df) > 0:
    if len(numeric_fields) > 0:
        # Histogram
        plt.figure(figsize=(8,4))
        main_df[numeric_field].hist(bins=30)
        plt.title(f"Histogram of '{numeric_field}'")
        plt.xlabel(numeric_field)
        plt.ylabel('Frequency')
        plt.show()

        # If possible, scatter plot numeric_field vs another numeric field
        if len(numeric_fields) > 1:
            plt.figure(figsize=(6,4))
            plt.scatter(main_df[numeric_fields[0]], main_df[numeric_fields[1]], alpha=0.5)
            plt.xlabel(numeric_fields[0])
            plt.ylabel(numeric_fields[1])
            plt.title(f"Scatter: {numeric_fields[0]} vs {numeric_fields[1]}")
            plt.show()
        
else:
    print("No visualization; DataFrame or numeric fields not available.")


## 6. Conclusion
This notebook demonstrates how to load and explore a dataset described by a Croissant schema using `mlcroissant`. You can use record set and field `@id`s to programmatically access or reference specific data within larger, more complex FAIR datasets.

Continue your exploration by updating the cell parameters with field `@id`s and analyzing the data relevant to your research!